# Notebook 15 ? Step 2 FM sequence-length ablation

Notebook ??c l?p cho Step 2: chu?n b? token cache, geometry kh?p model/sequence length, ch?y/resume 25 t? h?p v? t?ng h?p mean/std ri?ng t?ng test set. Hyperparameter l?y t? manifest Optuna Step 1; m?i t? h?p, k? c? 1001/501, ???c train l?i v?i geometry ESM1b/NTv1 kh?p. Cache token shard ?? ho?n t?t ???c gi? nguy?n v? t?i s? d?ng.


In [2]:
from pathlib import Path
import sys, json, os
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()), _cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Same data/cache/batch locations as Notebook 14; change only if intentionally using another batch.
BASE_DIR = Path(r"F:\variant_data")
POOLED_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "sequence_ablation"
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
FM_ABLATION_BATCH_RUN_DIR = REPO_ROOT / "experiments" / "fm_ablation_20261117_003149"
GEOMETRY_K = 64
STEP2_GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
STEP2_GEOMETRY_NORMALIZATION_ROOT = "normalization_artifacts_v2_step2_matched"
STEP2_GEOMETRY_INDEX_ROOT = "faiss_indexes_v2_step2_matched"
GEOM_FEATURE_NAMES = [f"{m}_{f}" for m in ("dna", "prot") for f in ("LVD_L2", "LVD_Cosine", "LID", "LVD_Relative")]
TARGET_CONFIGS = [{
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "geom_feature_names": GEOM_FEATURE_NAMES,
    "geometry_root": STEP2_GEOMETRY_ROOT_NAME,
}]
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
    {"name": "Train3Val_UniProt", "train": "train3", "val": "val", "test": "uniprot"},
]
PIPELINE_CONFIG = {"batch_size": 256, "epochs": 30, "lr": 1e-4, "weight_decay": 1e-3,
                   "lr_factor": 0.5, "lr_patience": 3, "early_stop_patience": 6,
                   "num_workers": 0, "seed": 42, "xgb_n_jobs": 8}
from core.module06_architecture_experiments.fm_ablation_preparation import build_split_configs
STEP2_SPLITS = build_split_configs(BASE_DIR)
STEP2_ROOT = FM_ABLATION_BATCH_RUN_DIR / "module06_architecture_experiments" / "step2_sequence_length_ablation"
print("Repository:", REPO_ROOT)
print("FM ablation batch:", FM_ABLATION_BATCH_RUN_DIR)
print("Token cache root:", TOKEN_CACHE_ROOT)
print("Geometry features:", GEOM_FEATURE_NAMES)
print("Splits:", [split["name"] for split in STEP2_SPLITS])


Repository: D:\Missense_variant_pathogenicity_prediction
FM ablation batch: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149
Token cache root: F:\variant_data\fm_embeddings\token_level_cache\module06
Geometry features: ['dna_LVD_L2', 'dna_LVD_Cosine', 'dna_LID', 'dna_LVD_Relative', 'prot_LVD_L2', 'prot_LVD_Cosine', 'prot_LID', 'prot_LVD_Relative']
Splits: ['train3', 'val', 'test', 'clinvarhq', 'proteingym', 'uniprot']


## Step 2 ? sequence-length ablation: token-cache preparation

Ch? d?ng NTv1 cho DNA v? ESM1b cho protein. Grid DNA l? 301/601/1001/2001/3501; protein l? 31/101/201/501/1001. Ri?ng DNA 3501, extractor ??c c?t chu?i 10001, c?t 3250 nucleotide m?i b?n quanh variant v? x?c nh?n variant ? gi?a c?a s? m?i (index 1750). File parquet g?c kh?ng b? s?a. Token cache resume theo shard; n?u model kh?ng h? tr? ?? 3501 nucleotide th? extraction d?ng thay v? c?t token ?m th?m.


In [3]:
# Token extraction for the selected FM pair and all requested lengths. Resume-safe; no training in this cell.
import importlib
import os
import pandas as pd
from IPython.display import display
from core.module06_architecture_experiments.fm_ablation_preparation import (
    inspect_token_cache, extract_missing_token_caches,
)
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    DNA_LENGTHS as STEP2_DNA_LENGTHS,
    PROTEIN_LENGTHS as STEP2_PROTEIN_LENGTHS,
    SEEDS as STEP2_SEEDS,
    build_selected_model_configs, verify_sequence_columns, build_center_geometry_embeddings,
)

STEP2_TOKEN_MODELS = build_selected_model_configs(STEP2_DNA_LENGTHS, STEP2_PROTEIN_LENGTHS)
STEP2_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / "step2_sequence_length_ablation"
STEP2_ROOT.mkdir(parents=True, exist_ok=True)
STEP2_COLUMN_AUDIT = pd.DataFrame(verify_sequence_columns(STEP2_SPLITS, STEP2_TOKEN_MODELS))
print(f"Requested sequence-length pairs: {len(STEP2_DNA_LENGTHS) * len(STEP2_PROTEIN_LENGTHS)}")
print("DNA lengths:", STEP2_DNA_LENGTHS, "| protein lengths:", STEP2_PROTEIN_LENGTHS)
print("Selected FM models: NTv1 DNA + ESM1b protein | seeds:", STEP2_SEEDS)
display(STEP2_COLUMN_AUDIT.groupby(["Model", "Sequence_Length", "Source_Sequence_Length", "Status"], as_index=False).size())
print("DNA 3501 crop: source columns ref_seq_10001/alt_seq_10001; crop 3250 nt on each side of the centered variant; variant index 1750.")

STEP2_CACHE_BEFORE = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP2_TOKEN_MODELS, STEP2_SPLITS))
print("Token cache inventory before extraction:")
display(STEP2_CACHE_BEFORE.groupby(["Model", "Status"], as_index=False).size())
os.environ["HF_HUB_OFFLINE"] = "1"
STEP2_TOKEN_CACHE_PREPARATION = extract_missing_token_caches(
    model_configs=STEP2_TOKEN_MODELS,
    split_configs=STEP2_SPLITS,
    cache_root=TOKEN_CACHE_ROOT,
    batch_run_dir=FM_ABLATION_BATCH_RUN_DIR,
    device="cuda",
    strict_context=True,  # Fail instead of silently truncating any requested full sequence.
)
STEP2_TOKEN_CACHE_INDEX = STEP2_TOKEN_CACHE_PREPARATION["index"]
STEP2_CACHE_AFTER = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP2_TOKEN_MODELS, STEP2_SPLITS))
display(STEP2_CACHE_AFTER.groupby(["Model", "Status"], as_index=False).size())
if (STEP2_CACHE_AFTER["Status"] != "complete").any():
    raise RuntimeError("Step 2 token-cache extraction is incomplete; rerun this cell to resume missing shards.")
print("Step 2 token cache index:", STEP2_TOKEN_CACHE_PREPARATION["index_path"])


Requested sequence-length pairs: 25
DNA lengths: (301, 601, 1001, 2001, 3501) | protein lengths: (31, 101, 201, 501, 1001)
Selected FM models: NTv1 DNA + ESM1b protein | seeds: (5, 6, 7, 37, 42)


,Model,Sequence_Length,Source_Sequence_Length,Status,size
0,esm1b_650m_prot_1001,1001,1001,ready,6
1,esm1b_650m_prot_101,101,101,ready,6
2,esm1b_650m_prot_201,201,201,ready,6
3,esm1b_650m_prot_31,31,31,ready,6
4,esm1b_650m_prot_501,501,501,ready,6
5,nt_v1_500m_dna_1001,1001,1001,ready,6
6,nt_v1_500m_dna_2001,2001,2001,ready,6
7,nt_v1_500m_dna_301,301,301,ready,6
8,nt_v1_500m_dna_3501,3501,10001,ready,6
9,nt_v1_500m_dna_601,601,601,ready,6


DNA 3501 crop: source columns ref_seq_10001/alt_seq_10001; crop 3250 nt on each side of the centered variant; variant index 1750.
Token cache inventory before extraction:


,Model,Status,size
0,esm1b_650m_prot_1001,complete,6
1,esm1b_650m_prot_101,complete,6
2,esm1b_650m_prot_201,complete,6
3,esm1b_650m_prot_31,complete,6
4,esm1b_650m_prot_501,complete,6
5,nt_v1_500m_dna_1001,complete,6
6,nt_v1_500m_dna_2001,complete,6
7,nt_v1_500m_dna_301,complete,6
8,nt_v1_500m_dna_3501,complete,6
9,nt_v1_500m_dna_601,complete,6


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Model,Status,size
0,esm1b_650m_prot_1001,complete,6
1,esm1b_650m_prot_101,complete,6
2,esm1b_650m_prot_201,complete,6
3,esm1b_650m_prot_31,complete,6
4,esm1b_650m_prot_501,complete,6
5,nt_v1_500m_dna_1001,complete,6
6,nt_v1_500m_dna_2001,complete,6
7,nt_v1_500m_dna_301,complete,6
8,nt_v1_500m_dna_3501,complete,6
9,nt_v1_500m_dna_601,complete,6


Step 2 token cache index: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\fm_ablation_token_cache_index.json


### Step 2 ? build model- and length-matched geometry

Geometry ???c t?nh t? center embedding ref/alt c?a ch?nh NTv1 DNA length v? ESM1b protein length t??ng ?ng. Cell d?n xu?t vector center t? token cache (kh?ng ch?y FM inference), r?i t?o LVD/LID geometry v?i k=64 v? chu?n h?a theo train3. Embedding trung gian, geometry, FAISS index v? scaler ??u ? th? m?c ri?ng; kh?ng ghi pooled cache d?ng cho Step 3.


In [4]:
# Derive center vectors for geometry only, then build/resume all matched k=64 geometry files.
from core.module06_architecture_experiments.fm_sequence_length_ablation import build_center_geometry_embeddings
from core.module03_geometry_extraction.geometry_v2 import GeometryV2Config, run_geometry_v2
import pyarrow.parquet as pq

STEP2_GEOMETRY_EMBEDDING_ROOT = BASE_DIR / "fm_embeddings" / "step2_geometry_center"
STEP2_GEOMETRY_MODELS = [model["name"] for model in STEP2_TOKEN_MODELS]
STEP2_CENTER_GEOMETRY_EMBEDDINGS = build_center_geometry_embeddings(
    model_configs=STEP2_TOKEN_MODELS,
    split_configs=STEP2_SPLITS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    output_root=STEP2_GEOMETRY_EMBEDDING_ROOT,
)
display(STEP2_CENTER_GEOMETRY_EMBEDDINGS.groupby(["Model", "Status"], as_index=False).size())

STEP2_GEOMETRY_CONFIG = GeometryV2Config(
    base_dir=str(BASE_DIR),
    embedding_dir=str(STEP2_GEOMETRY_EMBEDDING_ROOT),
    contexts=["train3"],
    splits=[split["name"] for split in STEP2_SPLITS],
    models=STEP2_GEOMETRY_MODELS,
    poolings=["center"],
    k_neighbors=GEOMETRY_K,
    resume=True,
    geometry_root=STEP2_GEOMETRY_ROOT_NAME,
    normalization_root=STEP2_GEOMETRY_NORMALIZATION_ROOT,
    index_root=STEP2_GEOMETRY_INDEX_ROOT,
)
STEP2_GEOMETRY_RUN = run_geometry_v2(STEP2_GEOMETRY_CONFIG)

STEP2_GEOMETRY_AUDIT = []
for _model in STEP2_TOKEN_MODELS:
    for _split in STEP2_SPLITS:
        _path = BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "train3" / _split["name"] / f"{_model['name']}_center_geom_v2_norm.parquet"
        if not _path.is_file():
            raise FileNotFoundError(f"Matched geometry file is missing: {_path}")
        _columns = set(pq.read_schema(_path).names)
        _required = {"Variant_ID", "LVD_L2", "LVD_Cosine", "LID", "LVD_Relative"}
        _missing = sorted(_required - _columns)
        if _missing:
            raise KeyError(f"Geometry file {_path} misses selected columns: {_missing}")
        STEP2_GEOMETRY_AUDIT.append({"Model": _model["name"], "Split": _split["name"], "Status": "ready", "Path": str(_path)})
STEP2_GEOMETRY_AUDIT = pd.DataFrame(STEP2_GEOMETRY_AUDIT)
display(STEP2_GEOMETRY_AUDIT.groupby("Model", as_index=False).agg(Splits=("Split", "nunique"), Status=("Status", "first")))
print("Matched geometry root:", BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "train3")


[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_2001/train3
[GEOM EMBEDDING RESUME] nt_v1_500m

,Model,Status,size
0,esm1b_650m_prot_1001,reused,6
1,esm1b_650m_prot_101,reused,6
2,esm1b_650m_prot_201,reused,6
3,esm1b_650m_prot_31,reused,6
4,esm1b_650m_prot_501,reused,6
5,nt_v1_500m_dna_1001,reused,6
6,nt_v1_500m_dna_2001,reused,6
7,nt_v1_500m_dna_301,reused,6
8,nt_v1_500m_dna_3501,reused,6
9,nt_v1_500m_dna_601,reused,6


[V2 RESUME] Skip train3/nt_v1_500m_dna_301_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_601_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_1001_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_2001_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_3501_center
[V2 RESUME] Skip train3/esm1b_650m_prot_31_center
[V2 RESUME] Skip train3/esm1b_650m_prot_101_center
[V2 RESUME] Skip train3/esm1b_650m_prot_201_center
[V2 RESUME] Skip train3/esm1b_650m_prot_501_center
[V2 RESUME] Skip train3/esm1b_650m_prot_1001_center


,Model,Splits,Status
0,esm1b_650m_prot_1001,6,ready
1,esm1b_650m_prot_101,6,ready
2,esm1b_650m_prot_201,6,ready
3,esm1b_650m_prot_31,6,ready
4,esm1b_650m_prot_501,6,ready
5,nt_v1_500m_dna_1001,6,ready
6,nt_v1_500m_dna_2001,6,ready
7,nt_v1_500m_dna_301,6,ready
8,nt_v1_500m_dna_3501,6,ready
9,nt_v1_500m_dna_601,6,ready


Matched geometry root: F:\variant_data\geometry_v2_k64_step2_matched\train3


## Step 2 ? train all 25 sequence-length pairs

M?i c?p d?ng geometry NTv1/ESM1b kh?p ??ng model v? sequence length, Cross-Attention 3 modality, c?a s? ref/alt 33 token, bio + SpliceAI, MLP v? 5 seed. Hyperparameter c?a t?ng seed l?y t? manifest Step 1. Checkpoint Step 1 ch? l? ngu?n hyperparameter; c? 25 c?p, bao g?m 1001/501, ??u ???c train l?i v? geometry ?? ??i. C?c run m?i h? tr? resume.


In [4]:
# Run/resume all 25 pairs ? five seeds. Step 1 checkpoints supply hyperparameters only.
import importlib
import json
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    load_source_seed_configs, run_sequence_length_ablation,
)

STEP2_SOURCE_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / "step1_optuna_nt_v1_dna1001_esm1b_prot501_5seed"
STEP2_SOURCE_CONFIGS, STEP2_SOURCE_MANIFESTS = load_source_seed_configs(
    STEP2_SOURCE_ROOT, STEP2_SEEDS
)
STEP2_SOURCE_CONFIG_TABLE = pd.DataFrame([
    {"Seed": seed, **config, "Source_Checkpoint": STEP2_SOURCE_MANIFESTS[seed]["checkpoint"]}
    for seed, config in STEP2_SOURCE_CONFIGS.items()
]).sort_values("Seed")
display(STEP2_SOURCE_CONFIG_TABLE)

if len(TARGET_CONFIGS) != 1 or TARGET_CONFIGS[0]["name"] != "B_with_geometry":
    raise ValueError("Step 2 is fixed to B_with_geometry only.")
if set(STEP2_TOKEN_CACHE_INDEX["models"]) != {model["name"] for model in STEP2_TOKEN_MODELS}:
    raise ValueError("Step 2 token-cache index does not cover exactly the requested NTv1/ESM1b length configs.")

# Reload the shared runner so the token-only sequence-ablation loader is active.
architecture_runner_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.architecture_experiment_runner"
))

def _step2_pipeline_factory(seed, dna_model, protein_model):
    return FusionBatchPipeline(
        base_dir=str(BASE_DIR),
        config={**PIPELINE_CONFIG, "seed": int(seed)},
        datasets=DATASETS,
        models_space=[dna_model, protein_model],
        pooling_strategies=["center"],
        experiments=[{"name": "PyTorch_CrossAttention_SequenceLength", "type": "pytorch", "fusion": "cross_attention"}],
        explainability={"enable_shap": False, "enable_lime": False},
        fm_profile_json=str(BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"),
        geom_profile_json=None,
        geometry_dir=str(BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "{context}"),
        geom_feature_names=GEOM_FEATURE_NAMES,
        batch_run_dir=str(FM_ABLATION_BATCH_RUN_DIR),
        embedding_dir=str(POOLED_CACHE_ROOT),
    )

STEP2_RUN_RESULT = run_sequence_length_ablation(
    pipeline_factory=_step2_pipeline_factory,
    target_configs=TARGET_CONFIGS,
    datasets=DATASETS,
    model_configs=STEP2_TOKEN_MODELS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    source_configs=STEP2_SOURCE_CONFIGS,
    source_manifests=STEP2_SOURCE_MANIFESTS,
    step_root=STEP2_ROOT,
    device="cuda",
    token_batch_size=64,
    display_fn=print,
)
print(STEP2_RUN_RESULT)


,Seed,attention_dropout,dropout,early_stop_patience,feature_dropout,ffn_multiplier,k_radius,learning_rate,lr_factor,lr_patience,max_epochs,model_dim,num_heads,num_layers,pooling,seed,weight_decay,Source_Checkpoint
0,5,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,5,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
1,6,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,6,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
2,7,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,7,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
3,37,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,37,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
4,42,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,42,0.000003,D:\Missense_variant_pathogenicity_prediction\e...


[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities_FM_SequenceLength] Train loader ready: 363 batches in 1.3s
[RESUME] Reusing CrossAttention_ThreeModalities_FM_SequenceLength checkpoint for B_with_geometry
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_ProteinGym
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_UniProt
CrossAttention_ThreeModalities_FM_SequenceLength training, validation, and test evaluation complete. 

In [5]:
# Run/resume the 1001-DNA / 501-protein pair on seeds 11-30.
# Reuse the fixed Step 1 hyperparameters (identical across the five source seeds);
# seed 42 is the provenance manifest, while every run gets its own seed/checkpoint folder.
import importlib
import pandas as pd
from IPython.display import display

_sequence_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.fm_sequence_length_ablation"
))
FOLLOWUP_SEEDS = tuple(range(11, 31))
FOLLOWUP_DNA_LENGTHS = (1001,)
FOLLOWUP_PROTEIN_LENGTHS = (501,)

# Fail early if either requested token cache is absent or incomplete.
FOLLOWUP_DNA_MODEL_NAME = "nt_v1_500m_dna_1001"
FOLLOWUP_PROTEIN_MODEL_NAME = "esm1b_650m_prot_501"
for _model_name in (FOLLOWUP_DNA_MODEL_NAME, FOLLOWUP_PROTEIN_MODEL_NAME):
    for _split in STEP2_SPLITS:
        _entry = STEP2_TOKEN_CACHE_INDEX["models"][_model_name][_split["name"]]
        if _entry.get("status") != "complete":
            raise RuntimeError(f"Required token cache is not complete: {_model_name}/{_split['name']}")

# Step 1 selected one fixed hyperparameter set shared by seeds 5, 6, 7, 37, and 42.
# Reuse that set for seeds 11-30; do not load or reuse seed-42 model weights.
FOLLOWUP_SOURCE_CONFIGS = {
    _seed: {**STEP2_SOURCE_CONFIGS[42], "seed": _seed}
    for _seed in FOLLOWUP_SEEDS
}
FOLLOWUP_SOURCE_MANIFESTS = {
    _seed: STEP2_SOURCE_MANIFESTS[42]
    for _seed in FOLLOWUP_SEEDS
}

FOLLOWUP_RUN_RESULT = _sequence_module.run_sequence_length_ablation(
    pipeline_factory=_step2_pipeline_factory,
    target_configs=TARGET_CONFIGS,
    datasets=DATASETS,
    model_configs=STEP2_TOKEN_MODELS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    source_configs=FOLLOWUP_SOURCE_CONFIGS,
    source_manifests=FOLLOWUP_SOURCE_MANIFESTS,
    step_root=STEP2_ROOT,
    device="cuda",
    token_batch_size=64,
    display_fn=print,
    dna_lengths=FOLLOWUP_DNA_LENGTHS,
    protein_lengths=FOLLOWUP_PROTEIN_LENGTHS,
    seeds=FOLLOWUP_SEEDS,
)
print(FOLLOWUP_RUN_RESULT)

# Collect this pair's per-seed validation and per-test-set results, then save
# mean/std separately for each test dataset.
FOLLOWUP_RUN_DIRS = [
    STEP2_ROOT / "runs" / f"step2_nt_v1_dna1001_esm1b_prot501_seed{_seed}"
    for _seed in FOLLOWUP_SEEDS
]
FOLLOWUP_VALIDATION = pd.concat([
    pd.read_csv(_run_dir / f"{_run_dir.name}_validation_metrics.csv")
    for _run_dir in FOLLOWUP_RUN_DIRS
    if (_run_dir / f"{_run_dir.name}_validation_metrics.csv").is_file()
], ignore_index=True, sort=False)
FOLLOWUP_TEST = pd.concat([
    pd.read_csv(_run_dir / f"{_run_dir.name}_test_metrics.csv")
    for _run_dir in FOLLOWUP_RUN_DIRS
    if (_run_dir / f"{_run_dir.name}_test_metrics.csv").is_file()
], ignore_index=True, sort=False)
if FOLLOWUP_VALIDATION.empty or FOLLOWUP_TEST.empty:
    raise FileNotFoundError("No completed validation/test metrics found for the 1001/501 follow-up.")

FOLLOWUP_VALIDATION["Seed"] = pd.to_numeric(FOLLOWUP_VALIDATION["Seed"], errors="coerce")
FOLLOWUP_TEST["Seed"] = pd.to_numeric(FOLLOWUP_TEST["Seed"], errors="coerce")
FOLLOWUP_VALIDATION = FOLLOWUP_VALIDATION.drop_duplicates(["Seed", "Target_Config"], keep="last").sort_values("Seed")
FOLLOWUP_TEST = FOLLOWUP_TEST.drop_duplicates(["Seed", "Target_Config", "Dataset"], keep="last").sort_values(["Dataset", "Seed"])
FOLLOWUP_VALIDATION.to_csv(STEP2_ROOT / "step2_1001_501_seed11_30_validation_by_seed.csv", index=False)
FOLLOWUP_TEST.to_csv(STEP2_ROOT / "step2_1001_501_seed11_30_test_by_seed.csv", index=False)

FOLLOWUP_METRICS = [
    "Accuracy", "Precision", "Recall", "Specificity", "F1_Score", "MCC", "AUROC", "AUPRC",
    "FM_GFLOPs/Sample", "FM_Latency_ms/Sample", "Geom_GFLOPs/Sample", "Geom_Latency_ms/Sample",
    "Classifier_GFLOPs/Sample", "Classifier_Latency_ms/Sample", "Classifier_Total_Params",
    "Classifier_Param_Mem_MB", "Classifier_Peak_VRAM_MB", "E2E_GFLOPs/Sample",
    "E2E_Latency_ms/Sample", "E2E_Total_Params", "E2E_Param_Mem_MB", "Max_Peak_VRAM_MB",
]

def _followup_mean_std(_frame, _group_columns, _metric_prefix=""):
    _metrics = [f"{_metric_prefix}{metric}" for metric in FOLLOWUP_METRICS if f"{_metric_prefix}{metric}" in _frame.columns]
    if not _metrics:
        raise ValueError(f"No follow-up metric columns found with prefix {_metric_prefix!r}; available columns: {list(_frame.columns)}")
    _result = _frame.groupby(_group_columns, dropna=False)[_metrics].agg(["mean", "std"])
    _result.columns = [f"{name}_{stat}" for name, stat in _result.columns]
    _result["N_Seeds"] = _frame.groupby(_group_columns, dropna=False)["Seed"].nunique()
    return _result.reset_index()

FOLLOWUP_VALIDATION_MEAN_STD = _followup_mean_std(FOLLOWUP_VALIDATION, ["Target_Config"], _metric_prefix="Val_")
FOLLOWUP_TEST_MEAN_STD = _followup_mean_std(FOLLOWUP_TEST, ["Target_Config", "Dataset"])
FOLLOWUP_VALIDATION_MEAN_STD.to_csv(STEP2_ROOT / "step2_1001_501_seed11_30_validation_mean_std.csv", index=False)
FOLLOWUP_TEST_MEAN_STD.to_csv(STEP2_ROOT / "step2_1001_501_seed11_30_test_mean_std_by_dataset.csv", index=False)
print("Validation results by seed:")
display(FOLLOWUP_VALIDATION[[c for c in ["Seed", "Target_Config", *(f"Val_{metric}" for metric in FOLLOWUP_METRICS)] if c in FOLLOWUP_VALIDATION.columns]])
print("Validation mean/std across available seeds:")
display(FOLLOWUP_VALIDATION_MEAN_STD)
for _dataset, _rows in FOLLOWUP_TEST_MEAN_STD.groupby("Dataset", sort=False):
    print(f"Test results: {_dataset}")
    display(_rows)
print("Saved per-seed and mean/std CSV files under:", STEP2_ROOT)


[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities_FM_SequenceLength] Train loader ready: 363 batches in 1.3s
[RESUME] Reusing CrossAttention_ThreeModalities_FM_SequenceLength checkpoint for B_with_geometry
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_ProteinGym
[CrossAttention_ThreeModalities_FM_SequenceLength] Reusing saved test metrics: B_with_geometry / Train3Val_UniProt
CrossAttention_ThreeModalities_FM_SequenceLength training, validation, and test evaluation complete. 

,Seed,Target_Config,Val_Accuracy,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_MCC,Val_AUROC,Val_AUPRC
0,11,B_with_geometry,0.9456,0.8764,0.8324,0.9723,0.8538,0.8209,0.9756,0.9229
1,12,B_with_geometry,0.9418,0.8498,0.8443,0.9648,0.8471,0.8111,0.9735,0.9155
2,13,B_with_geometry,0.9423,0.8737,0.8152,0.9722,0.8435,0.8088,0.9741,0.9182
3,14,B_with_geometry,0.9429,0.8425,0.8615,0.9620,0.8519,0.8166,0.9735,0.9159
4,15,B_with_geometry,0.9449,0.8774,0.8267,0.9728,0.8513,0.8181,0.9744,0.9191
5,16,B_with_geometry,0.9439,0.8542,0.8516,0.9657,0.8529,0.8183,0.9743,0.9212
6,17,B_with_geometry,0.9415,0.8556,0.8344,0.9668,0.8449,0.8089,0.9729,0.9177
7,18,B_with_geometry,0.9417,0.8217,0.8868,0.9546,0.8530,0.8176,0.9752,0.9215
8,19,B_with_geometry,0.9425,0.8456,0.8546,0.9632,0.8501,0.8145,0.9716,0.9146
9,20,B_with_geometry,0.9439,0.8588,0.8447,0.9673,0.8517,0.8171,0.9742,0.9199


Validation mean/std across available seeds:


,Target_Config,Val_Accuracy_mean,Val_Accuracy_std,Val_Precision_mean,Val_Precision_std,Val_Recall_mean,Val_Recall_std,Val_Specificity_mean,Val_Specificity_std,Val_F1_Score_mean,Val_F1_Score_std,Val_MCC_mean,Val_MCC_std,Val_AUROC_mean,Val_AUROC_std,Val_AUPRC_mean,Val_AUPRC_std,N_Seeds
0,B_with_geometry,0.942755,0.001318,0.855435,0.016708,0.843215,0.018566,0.966235,0.00526,0.84895,0.00317,0.813925,0.003674,0.97359,0.001286,0.918245,0.00257,20


Test results: Train3Val_ClinVarHQ


,Target_Config,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
0,B_with_geometry,Train3Val_ClinVarHQ,0.933075,0.00821,0.98028,0.008377,0.88276,0.021915,0.982385,0.008008,...,0.0,35.54,0.010761,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,20


... (3 outputs left out)

,Target_Config,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
2,B_with_geometry,Train3Val_Test,0.932135,0.003092,0.88589,0.013679,0.862065,0.020272,0.95832,0.006445,...,0.0,34.641,0.004472,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,20


Test results: Train3Val_UniProt


,Target_Config,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
3,B_with_geometry,Train3Val_UniProt,0.759185,0.010079,0.752675,0.013429,0.770105,0.02372,0.748345,0.022357,...,0.0,35.108,0.009515,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,20


Saved per-seed and mean/std CSV files under: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\step2_sequence_length_ablation


In [4]:
# Run/resume the 1001-DNA / 501-protein pair on seeds 11-30.
# Reuse the fixed Step 1 hyperparameters (identical across the five source seeds);
# seed 42 is the provenance manifest, while every run gets its own seed/checkpoint folder.
import importlib
import pandas as pd
from IPython.display import display
# Run/resume all 25 pairs ? five seeds. Step 1 checkpoints supply hyperparameters only.
import importlib
import json
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    load_source_seed_configs, run_sequence_length_ablation,
)

STEP2_SOURCE_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / "step1_optuna_nt_v1_dna1001_esm1b_prot501_5seed"
STEP2_SOURCE_CONFIGS, STEP2_SOURCE_MANIFESTS = load_source_seed_configs(
    STEP2_SOURCE_ROOT, STEP2_SEEDS
)
STEP2_SOURCE_CONFIG_TABLE = pd.DataFrame([
    {"Seed": seed, **config, "Source_Checkpoint": STEP2_SOURCE_MANIFESTS[seed]["checkpoint"]}
    for seed, config in STEP2_SOURCE_CONFIGS.items()
]).sort_values("Seed")
display(STEP2_SOURCE_CONFIG_TABLE)

if len(TARGET_CONFIGS) != 1 or TARGET_CONFIGS[0]["name"] != "B_with_geometry":
    raise ValueError("Step 2 is fixed to B_with_geometry only.")
if set(STEP2_TOKEN_CACHE_INDEX["models"]) != {model["name"] for model in STEP2_TOKEN_MODELS}:
    raise ValueError("Step 2 token-cache index does not cover exactly the requested NTv1/ESM1b length configs.")

# Reload the shared runner so the token-only sequence-ablation loader is active.
architecture_runner_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.architecture_experiment_runner"
))

def _step2_pipeline_factory(seed, dna_model, protein_model):
    return FusionBatchPipeline(
        base_dir=str(BASE_DIR),
        config={**PIPELINE_CONFIG, "seed": int(seed)},
        datasets=DATASETS,
        models_space=[dna_model, protein_model],
        pooling_strategies=["center"],
        experiments=[{"name": "PyTorch_CrossAttention_SequenceLength", "type": "pytorch", "fusion": "cross_attention"}],
        explainability={"enable_shap": False, "enable_lime": False},
        fm_profile_json=str(BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"),
        geom_profile_json=None,
        geometry_dir=str(BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "{context}"),
        geom_feature_names=GEOM_FEATURE_NAMES,
        batch_run_dir=str(FM_ABLATION_BATCH_RUN_DIR),
        embedding_dir=str(POOLED_CACHE_ROOT),
    )
_sequence_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.fm_sequence_length_ablation"
))
FOLLOWUP_SEEDS = tuple(range(51, 71))
FOLLOWUP_DNA_LENGTHS = (1001,)
FOLLOWUP_PROTEIN_LENGTHS = (501,)

# Fail early if either requested token cache is absent or incomplete.
FOLLOWUP_DNA_MODEL_NAME = "nt_v1_500m_dna_1001"
FOLLOWUP_PROTEIN_MODEL_NAME = "esm1b_650m_prot_501"
for _model_name in (FOLLOWUP_DNA_MODEL_NAME, FOLLOWUP_PROTEIN_MODEL_NAME):
    for _split in STEP2_SPLITS:
        _entry = STEP2_TOKEN_CACHE_INDEX["models"][_model_name][_split["name"]]
        if _entry.get("status") != "complete":
            raise RuntimeError(f"Required token cache is not complete: {_model_name}/{_split['name']}")

# Step 1 selected one fixed hyperparameter set shared by seeds 5, 6, 7, 37, and 42.
# Reuse that set for seeds 11-30; do not load or reuse seed-42 model weights.
FOLLOWUP_SOURCE_CONFIGS = {
    _seed: {**STEP2_SOURCE_CONFIGS[42], "seed": _seed}
    for _seed in FOLLOWUP_SEEDS
}
FOLLOWUP_SOURCE_MANIFESTS = {
    _seed: STEP2_SOURCE_MANIFESTS[42]
    for _seed in FOLLOWUP_SEEDS
}

FOLLOWUP_RUN_RESULT = _sequence_module.run_sequence_length_ablation(
    pipeline_factory=_step2_pipeline_factory,
    target_configs=TARGET_CONFIGS,
    datasets=DATASETS,
    model_configs=STEP2_TOKEN_MODELS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    source_configs=FOLLOWUP_SOURCE_CONFIGS,
    source_manifests=FOLLOWUP_SOURCE_MANIFESTS,
    step_root=STEP2_ROOT,
    device="cuda",
    token_batch_size=64,
    display_fn=print,
    dna_lengths=FOLLOWUP_DNA_LENGTHS,
    protein_lengths=FOLLOWUP_PROTEIN_LENGTHS,
    seeds=FOLLOWUP_SEEDS,
)
print(FOLLOWUP_RUN_RESULT)

# Collect this pair's per-seed validation and per-test-set results, then save
# mean/std separately for each test dataset.
FOLLOWUP_RUN_DIRS = [
    STEP2_ROOT / "runs" / f"step2_nt_v1_dna1001_esm1b_prot501_seed{_seed}"
    for _seed in FOLLOWUP_SEEDS
]
FOLLOWUP_VALIDATION = pd.concat([
    pd.read_csv(_run_dir / f"{_run_dir.name}_validation_metrics.csv")
    for _run_dir in FOLLOWUP_RUN_DIRS
    if (_run_dir / f"{_run_dir.name}_validation_metrics.csv").is_file()
], ignore_index=True, sort=False)
FOLLOWUP_TEST = pd.concat([
    pd.read_csv(_run_dir / f"{_run_dir.name}_test_metrics.csv")
    for _run_dir in FOLLOWUP_RUN_DIRS
    if (_run_dir / f"{_run_dir.name}_test_metrics.csv").is_file()
], ignore_index=True, sort=False)
if FOLLOWUP_VALIDATION.empty or FOLLOWUP_TEST.empty:
    raise FileNotFoundError("No completed validation/test metrics found for the 1001/501 follow-up.")

FOLLOWUP_VALIDATION["Seed"] = pd.to_numeric(FOLLOWUP_VALIDATION["Seed"], errors="coerce")
FOLLOWUP_TEST["Seed"] = pd.to_numeric(FOLLOWUP_TEST["Seed"], errors="coerce")
FOLLOWUP_VALIDATION = FOLLOWUP_VALIDATION.drop_duplicates(["Seed", "Target_Config"], keep="last").sort_values("Seed")
FOLLOWUP_TEST = FOLLOWUP_TEST.drop_duplicates(["Seed", "Target_Config", "Dataset"], keep="last").sort_values(["Dataset", "Seed"])
FOLLOWUP_VALIDATION.to_csv(STEP2_ROOT / "step2_1001_501_seed51_70_validation_by_seed.csv", index=False)
FOLLOWUP_TEST.to_csv(STEP2_ROOT / "step2_1001_501_seed51_70_test_by_seed.csv", index=False)

FOLLOWUP_METRICS = [
    "Accuracy", "Precision", "Recall", "Specificity", "F1_Score", "MCC", "AUROC", "AUPRC",
    "FM_GFLOPs/Sample", "FM_Latency_ms/Sample", "Geom_GFLOPs/Sample", "Geom_Latency_ms/Sample",
    "Classifier_GFLOPs/Sample", "Classifier_Latency_ms/Sample", "Classifier_Total_Params",
    "Classifier_Param_Mem_MB", "Classifier_Peak_VRAM_MB", "E2E_GFLOPs/Sample",
    "E2E_Latency_ms/Sample", "E2E_Total_Params", "E2E_Param_Mem_MB", "Max_Peak_VRAM_MB",
]

def _followup_mean_std(_frame, _group_columns, _metric_prefix=""):
    _metrics = [f"{_metric_prefix}{metric}" for metric in FOLLOWUP_METRICS if f"{_metric_prefix}{metric}" in _frame.columns]
    if not _metrics:
        raise ValueError(f"No follow-up metric columns found with prefix {_metric_prefix!r}; available columns: {list(_frame.columns)}")
    _result = _frame.groupby(_group_columns, dropna=False)[_metrics].agg(["mean", "std"])
    _result.columns = [f"{name}_{stat}" for name, stat in _result.columns]
    _result["N_Seeds"] = _frame.groupby(_group_columns, dropna=False)["Seed"].nunique()
    return _result.reset_index()

FOLLOWUP_VALIDATION_MEAN_STD = _followup_mean_std(FOLLOWUP_VALIDATION, ["Target_Config"], _metric_prefix="Val_")
FOLLOWUP_TEST_MEAN_STD = _followup_mean_std(FOLLOWUP_TEST, ["Target_Config", "Dataset"])
FOLLOWUP_VALIDATION_MEAN_STD.to_csv(STEP2_ROOT / "step2_1001_501_seed51_70_validation_mean_std.csv", index=False)
FOLLOWUP_TEST_MEAN_STD.to_csv(STEP2_ROOT / "step2_1001_501_seed51_70_test_mean_std_by_dataset.csv", index=False)
print("Validation results by seed:")
display(FOLLOWUP_VALIDATION[[c for c in ["Seed", "Target_Config", *(f"Val_{metric}" for metric in FOLLOWUP_METRICS)] if c in FOLLOWUP_VALIDATION.columns]])
print("Validation mean/std across available seeds:")
display(FOLLOWUP_VALIDATION_MEAN_STD)
for _dataset, _rows in FOLLOWUP_TEST_MEAN_STD.groupby("Dataset", sort=False):
    print(f"Test results: {_dataset}")
    display(_rows)
print("Saved per-seed and mean/std CSV files under:", STEP2_ROOT)


,Seed,attention_dropout,dropout,early_stop_patience,feature_dropout,ffn_multiplier,k_radius,learning_rate,lr_factor,lr_patience,max_epochs,model_dim,num_heads,num_layers,pooling,seed,weight_decay,Source_Checkpoint
0,5,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,5,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
1,6,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,6,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
2,7,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,7,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
3,37,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,37,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
4,42,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,42,0.000003,D:\Missense_variant_pathogenicity_prediction\e...


[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 11604/11604 [00:06<00:00, 1661.23shard/s]


[Window cache] dna/train3: 23208 samples ready in 7.0s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:01<00:00, 845.47shard/s]


[Window cache] protein/train3: 23208 samples ready in 1.7s
[CrossAttention_ThreeModalities_FM_SequenceLength] Train loader ready: 363 batches in 10.2s
[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing validation dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



... (906 outputs left out)

,Target_Config,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
2,B_with_geometry,Train3Val_Test,0.932435,0.002448,0.88902,0.011108,0.859195,0.017833,0.95978,0.005227,...,0.0,34.6405,0.00394,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,20


Test results: Train3Val_UniProt


,Target_Config,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
3,B_with_geometry,Train3Val_UniProt,0.75956,0.007462,0.75533,0.007457,0.765435,0.020788,0.753735,0.013295,...,0.0,35.104,0.008208,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,20


Saved per-seed and mean/std CSV files under: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\step2_sequence_length_ablation


## Step 2 ? results and per-test-set mean/std

Cell t?ng h?p k?t qu? c?a c?c run geometry-matched m?i cho ?? 25 t? h?p. Ch?n c?u h?nh theo validation MCC; b?o c?o ri?ng t?ng test set, kh?ng g?p b?n test set. C?c CSV l?u metric, profiling, seed v? hyperparameter t?ng run c?ng mean/std.


In [5]:
# Read completed geometry-matched Step 2 rows and save full and mean/std tables.
from IPython.display import display
from core.module06_architecture_experiments.fm_sequence_length_ablation import summarize_sequence_length_ablation

STEP2_SUMMARY = summarize_sequence_length_ablation(
    step_root=STEP2_ROOT,
    seeds=STEP2_SEEDS,
)
print("Validation mean/std across seeds; ranking is by validation MCC:")
display(STEP2_SUMMARY["validation_mean_std"])

STEP2_TEST_DISPLAY_COLS = [
    "DNA_Length", "Protein_Length", "N_Seeds",
    "Accuracy_mean", "Accuracy_std", "Precision_mean", "Precision_std",
    "Recall_mean", "Recall_std", "Specificity_mean", "Specificity_std",
    "F1_Score_mean", "F1_Score_std", "MCC_mean", "MCC_std",
    "AUROC_mean", "AUROC_std", "AUPRC_mean", "AUPRC_std",
    "FM_GFLOPs/Sample_mean", "FM_Latency_ms/Sample_mean",
    "Geom_GFLOPs/Sample_mean", "Geom_Latency_ms/Sample_mean",
    "Classifier_GFLOPs/Sample_mean", "Classifier_Latency_ms/Sample_mean",
    "Classifier_Total_Params_mean", "Classifier_Param_Mem_MB_mean",
    "E2E_GFLOPs/Sample_mean", "E2E_Latency_ms/Sample_mean",
    "E2E_Total_Params_mean", "E2E_Param_Mem_MB_mean", "Max_Peak_VRAM_MB_mean",
]
for _dataset_name, _dataset_frame in STEP2_SUMMARY["test_mean_std_by_dataset"].groupby("Dataset", sort=False):
    _columns = [column for column in STEP2_TEST_DISPLAY_COLS if column in _dataset_frame.columns]
    print(f"\nTest set: {_dataset_name} (sorted by mean MCC)")
    display(_dataset_frame[_columns].sort_values("MCC_mean", ascending=False).reset_index(drop=True))

print("Combo completion inventory:")
display(STEP2_SUMMARY["inventory"].sort_values(["DNA_Length", "Protein_Length"]).reset_index(drop=True))
print("Saved detailed and aggregate CSVs under:", STEP2_SUMMARY["output_root"])


Validation mean/std across seeds; ranking is by validation MCC:


,DNA_Length,Protein_Length,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Decision_Threshold_mean,Decision_Threshold_std,Val_MCC_mean,Val_MCC_std,...,Val_Recall_std,Val_Specificity_mean,Val_Specificity_std,Val_F1_Score_mean,Val_F1_Score_std,Val_AUROC_mean,Val_AUROC_std,Val_AUPRC_mean,Val_AUPRC_std,N_Seeds
13,1001,501,2.4,0.894427,134.405688,13.244057,0.731869,0.199707,0.82270,0.001623,...,0.010882,0.96890,0.003085,0.85602,0.001506,0.97544,0.000195,0.92260,0.000866,5
17,2001,201,2.4,0.894427,153.653503,16.543091,0.672764,0.187488,0.82236,0.003833,...,0.006271,0.96624,0.002596,0.85620,0.002956,0.97544,0.000680,0.92224,0.002495,5
22,3501,201,2.2,0.447214,149.010893,7.860231,0.690866,0.131536,0.82188,0.004533,...,0.007319,0.96944,0.002887,0.85526,0.003467,0.97530,0.000771,0.92152,0.002848,5
2,301,201,2.2,0.447214,118.742369,6.369710,0.711263,0.136277,0.82122,0.002190,...,0.010100,0.97034,0.003100,0.85448,0.001785,0.97514,0.000847,0.92202,0.002696,5
7,601,201,2.4,0.894427,145.670205,15.485724,0.690992,0.189742,0.82118,0.003425,...,0.006011,0.96662,0.001550,0.85516,0.002811,0.97538,0.000823,0.92188,0.002872,5
12,1001,201,2.4,0.894427,150.419716,15.843041,0.668636,0.150132,0.82052,0.003920,...,0.014440,0.96536,0.005676,0.85464,0.002659,0.97518,0.000983,0.92126,0.003595,5
4,301,1001,1.8,0.447214,118.274433,1.255243,0.622603,0.170627,0.81920,0.003317,...,0.010064,0.96702,0.002925,0.85340,0.002741,0.97444,0.001234,0.92018,0.003098,5
14,1001,1001,2.0,0.000000,145.673140,0.230548,0.672915,0.178069,0.81910,0.004486,...,0.015503,0.96734,0.004004,0.85320,0.003952,0.97440,0.001022,0.91932,0.003604,5
23,3501,501,2.4,0.894427,151.953909,15.622828,0.688183,0.204933,0.81904,0.004179,...,0.012128,0.96646,0.003195,0.85336,0.003415,0.97480,0.000735,0.92038,0.002587,5
24,3501,1001,1.8,0.447214,141.269515,7.884190,0.631143,0.205119,0.81902,0.003927,...,0.017374,0.96726,0.003982,0.85316,0.003663,0.97462,0.001089,0.91954,0.003678,5



Test set: Train3Val_ClinVarHQ (sorted by mean MCC)


,DNA_Length,Protein_Length,N_Seeds,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,...,Geom_Latency_ms/Sample_mean,Classifier_GFLOPs/Sample_mean,Classifier_Latency_ms/Sample_mean,Classifier_Total_Params_mean,Classifier_Param_Mem_MB_mean,E2E_GFLOPs/Sample_mean,E2E_Latency_ms/Sample_mean,E2E_Total_Params_mean,E2E_Param_Mem_MB_mean,Max_Peak_VRAM_MB_mean
0,3501,1001,5,0.94226,0.006856,0.97896,0.007574,0.90288,0.019631,0.98082,...,1.12,1.3669,0.180,33273860.0,126.93,1306.9526,61.210,6.856330e+08,1371.21,2132.78
1,1001,1001,5,0.94110,0.007667,0.98008,0.008197,0.89944,0.017355,0.98198,...,1.12,1.3669,0.180,33273860.0,126.93,1471.7314,67.810,1.171332e+09,2297.61,2132.78
2,2001,201,5,0.94054,0.002737,0.97464,0.007545,0.90342,0.004345,0.97688,...,1.12,1.3669,0.180,33273860.0,126.93,588.4595,26.030,1.171332e+09,2297.61,1957.27
3,3501,501,5,0.94024,0.005713,0.97536,0.009016,0.90230,0.017500,0.97746,...,1.12,1.3669,0.180,33273860.0,126.93,656.1123,28.970,6.856330e+08,1371.21,2132.94
4,601,501,5,0.93998,0.008057,0.97482,0.011642,0.90228,0.022997,0.97690,...,1.12,1.3669,0.180,33273860.0,126.93,753.8299,32.950,1.171332e+09,2297.61,2132.94
5,301,1001,5,0.93940,0.005381,0.97698,0.005212,0.89884,0.015510,0.97914,...,1.12,1.3669,0.160,33273860.0,126.93,1356.7694,63.150,1.171332e+09,2297.61,2132.78
6,2001,1001,5,0.93882,0.009273,0.98008,0.010077,0.89484,0.023575,0.98196,...,1.12,1.3669,0.180,33273860.0,126.93,1629.8040,74.690,1.171332e+09,2297.61,2132.78
7,1001,501,5,0.93882,0.005767,0.98062,0.008192,0.89424,0.015641,0.98252,...,1.12,1.3669,0.162,33273860.0,126.93,820.8911,35.552,1.171332e+09,2297.61,2132.94
8,601,1001,5,0.93798,0.003825,0.97632,0.007172,0.89656,0.011669,0.97856,...,1.12,1.3669,0.180,33273860.0,126.93,1404.6702,65.190,1.171332e+09,2297.61,2132.78
9,1001,201,5,0.93772,0.003678,0.97102,0.008193,0.90116,0.012612,0.97352,...,1.12,1.3669,0.180,33273860.0,126.93,430.3869,19.150,1.171332e+09,2297.61,1957.27



Test set: Train3Val_ProteinGym (sorted by mean MCC)


,DNA_Length,Protein_Length,N_Seeds,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,...,Geom_Latency_ms/Sample_mean,Classifier_GFLOPs/Sample_mean,Classifier_Latency_ms/Sample_mean,Classifier_Total_Params_mean,Classifier_Param_Mem_MB_mean,E2E_GFLOPs/Sample_mean,E2E_Latency_ms/Sample_mean,E2E_Total_Params_mean,E2E_Param_Mem_MB_mean,Max_Peak_VRAM_MB_mean
0,1001,1001,5,0.84374,0.007830,0.78570,0.012600,0.90476,0.007068,0.79248,...,0.97,1.3669,0.180,33273860.0,126.93,1471.7314,67.170,1.171332e+09,2297.61,2132.78
1,3501,1001,5,0.84020,0.010427,0.77900,0.017315,0.90830,0.014636,0.78300,...,0.97,1.3669,0.180,33273860.0,126.93,1306.9526,60.680,6.856330e+08,1371.21,2132.78
2,301,1001,5,0.84008,0.010864,0.77892,0.017203,0.90802,0.014058,0.78302,...,0.97,1.3669,0.182,33273860.0,126.93,1356.7694,62.622,1.171332e+09,2297.61,2132.78
3,2001,1001,5,0.84022,0.011021,0.78080,0.020933,0.90506,0.013939,0.78576,...,0.97,1.3669,0.180,33273860.0,126.93,1629.8040,73.800,1.171332e+09,2297.61,2132.78
4,601,1001,5,0.84008,0.009901,0.78072,0.013305,0.90386,0.011515,0.78648,...,0.97,1.3669,0.180,33273860.0,126.93,1404.6702,64.540,1.171332e+09,2297.61,2132.78
5,301,201,5,0.84008,0.008506,0.78270,0.008194,0.89942,0.010069,0.79022,...,0.97,1.3669,0.180,33273860.0,126.93,315.4249,14.130,1.171332e+09,2297.61,1957.27
6,2001,201,5,0.83752,0.006654,0.77394,0.010651,0.91012,0.004290,0.77652,...,0.97,1.3669,0.180,33273860.0,126.93,588.4595,25.310,1.171332e+09,2297.61,1957.27
7,601,201,5,0.83628,0.005605,0.77334,0.011293,0.90774,0.009495,0.77624,...,0.97,1.3669,0.180,33273860.0,126.93,363.3257,16.050,1.171332e+09,2297.61,1957.27
8,2001,501,5,0.83724,0.006675,0.77918,0.013224,0.89852,0.011796,0.78574,...,0.97,1.3669,0.180,33273860.0,126.93,978.9637,41.690,1.171332e+09,2297.61,2132.94
9,601,101,5,0.83722,0.010828,0.78044,0.021674,0.89672,0.016254,0.78728,...,0.97,1.3669,0.180,33273860.0,126.93,233.1576,10.590,1.171332e+09,2297.61,1962.76


... (3 outputs left out)

,DNA_Length,Protein_Length,N_Seeds,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,...,Geom_Latency_ms/Sample_mean,Classifier_GFLOPs/Sample_mean,Classifier_Latency_ms/Sample_mean,Classifier_Total_Params_mean,Classifier_Param_Mem_MB_mean,E2E_GFLOPs/Sample_mean,E2E_Latency_ms/Sample_mean,E2E_Total_Params_mean,E2E_Param_Mem_MB_mean,Max_Peak_VRAM_MB_mean
0,2001,501,5,0.77118,0.005929,0.76396,0.007071,0.78250,0.013085,0.75992,...,0.96,1.3669,0.180,33273860.0,126.93,978.9637,41.800,1.171332e+09,2297.61,2132.94
1,1001,1001,5,0.76952,0.005534,0.75726,0.010002,0.79126,0.015443,0.74798,...,0.96,1.3669,0.180,33273860.0,126.93,1471.7314,67.200,1.171332e+09,2297.61,2132.78
2,301,201,5,0.76952,0.008245,0.76492,0.008882,0.77588,0.015021,0.76324,...,0.96,1.3669,0.140,33273860.0,126.93,315.4249,14.370,1.171332e+09,2297.61,1957.27
3,1001,201,5,0.76892,0.005469,0.75702,0.011203,0.79006,0.012711,0.74798,...,0.96,1.3669,0.180,33273860.0,126.93,430.3869,18.950,1.171332e+09,2297.61,1957.27
4,2001,201,5,0.76894,0.001677,0.75684,0.003658,0.79006,0.009276,0.74798,...,0.96,1.3669,0.180,33273860.0,126.93,588.4595,25.610,1.171332e+09,2297.61,1957.27
5,601,501,5,0.76848,0.010392,0.75882,0.005119,0.78466,0.027647,0.75246,...,0.96,1.3669,0.180,33273860.0,126.93,753.8299,32.480,1.171332e+09,2297.61,2132.94
6,3501,501,5,0.76816,0.008232,0.75800,0.012224,0.78584,0.014014,0.75066,...,0.96,1.3669,0.180,33273860.0,126.93,656.1123,28.590,6.856330e+08,1371.21,2132.94
7,601,201,5,0.76818,0.002045,0.75706,0.002991,0.78736,0.009903,0.74918,...,0.96,1.3669,0.180,33273860.0,126.93,363.3257,16.290,1.171332e+09,2297.61,1957.27
8,1001,501,5,0.76728,0.008967,0.76136,0.008520,0.77620,0.018729,0.75844,...,0.96,1.3669,0.164,33273860.0,126.93,820.8911,35.124,1.171332e+09,2297.61,2132.94
9,601,1001,5,0.76698,0.011175,0.75536,0.008376,0.78704,0.018505,0.74708,...,0.96,1.3669,0.178,33273860.0,126.93,1404.6702,64.538,1.171332e+09,2297.61,2132.78


Combo completion inventory:


,DNA_Length,Protein_Length,Validation_Seeds,Test_Seeds,Test_Rows,All_Seeds_Have_All_Four_Test_Sets
0,301,31,5,5,20,True
1,301,101,5,5,20,True
2,301,201,5,5,20,True
3,301,501,5,5,20,True
4,301,1001,5,5,20,True
5,601,31,5,5,20,True
6,601,101,5,5,20,True
7,601,201,5,5,20,True
8,601,501,5,5,20,True
9,601,1001,5,5,20,True


Saved detailed and aggregate CSVs under: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\step2_sequence_length_ablation
